# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset details protein abundance, post-translational modifications, coverage, and sequencing summary metrics obtained via mass spectrometry from human mast cell extracellular vesicles.

### Dataset Source
This dataset is defined by a Croissant schema URL and is published on [SEN Science](https://sen.science/).

**Croissant schema URL:**  
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

> **Note:** All references to entities correspond to their `@id` field as defined in the dataset schema.

In [ ]:
# List record sets and their fields (@id)
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}\n")
    field_ids = [field.id for field in rs.fields]
    print("  Fields (@id):")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name}")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @id values (for flexible exploration)
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load each record set as a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id} with {len(df)} records and columns:")
        print(df.columns.tolist())
        print(df.head(2), "\n")
    else:
        print(f"No records found for record set: {record_set_id}")

# For demonstration, select the "main" record set (often with most fields) for next steps
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
main_df = dataframes[main_record_set_id] if main_record_set_id in dataframes else None

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming numeric columns, and grouping by categorical variables for meaningful insights.

In [ ]:
import numpy as np

if main_df is not None and not main_df.empty:
    print(f'Columns in main DataFrame: {list(main_df.columns)}')
    
    # Attempt to auto-select a numeric field (e.g., any column with float/int values)
    numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
    if len(numeric_fields) == 0:
        # Try to convert potential numeric columns
        for col in main_df.columns:
            try:
                main_df[col] = pd.to_numeric(main_df[col], errors='ignore')
            except Exception:
                pass
        numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]

    print(f"Numeric fields detected: {numeric_fields}\n")
    
    numeric_field = numeric_fields[0] if numeric_fields else None
    if numeric_field:
        # Filtering
        threshold = main_df[numeric_field].quantile(0.75)
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records in '{main_record_set_id}' with {numeric_field} > {threshold:.3f} (75th percentile):")
        print(filtered_df[[numeric_field]].head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to identify a grouping/categorical field
        object_fields = [col for col in main_df.columns if main_df[col].dtype == object and main_df[col].nunique() < 10]
        group_field = object_fields[0] if object_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by '{group_field}' (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable group/categorical field found.")
    else:
        print("No numeric fields could be identified for EDA.")
else:
    print("No main DataFrame available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. This section demonstrates plotting a histogram for a numeric field and (where possible) a boxplot by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion

- The dataset was loaded via its FAIR Croissant schema and explored using `mlcroissant`.
- We reviewed available record sets, fields, and extracted the main data as a DataFrame, referencing all entities by their `@id`.
- Exploratory analysis was applied to numeric fields, including filtering, normalization, and grouping.
- Visualization illustrates the characteristics of main quantitative field(s) in the dataset.

For further analysis, consult the published [dataset record](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and the Croissant schema for full details on field definitions, methodology, and downstream use.